In [0]:
schema_name = dbutils.widgets.get("schema_param")
table_name = dbutils.widgets.get("table_param")

In [0]:
import os
import pyspark.sql.functions as F

df = spark.read.parquet(f"{os.getcwd()}/flights-1m.parquet")

In [0]:
spark.sql("USE CATALOG workspace;")

In [0]:
df_cleaned = df.withColumn("FL_DATE", F.to_date(F.col("FL_DATE"), "yyyy-MM-dd"))

df_transformed = df_cleaned \
    .withColumn("TOTAL_DELAY", F.col("DEP_DELAY") + F.col("ARR_DELAY")) \
    .withColumn("IS_DELAYED", F.when(F.col("ARR_DELAY") > 0, 1).otherwise(0)) \
    .withColumn("AVG_SPEED", F.round(F.col("DISTANCE") / (F.col("AIR_TIME") / 60), 2))

df_filtered = df_transformed.filter((F.col("ARR_DELAY") > 0) & (F.col("DISTANCE") > 2000))

df_final = df_transformed.groupBy("FL_DATE").agg(
    F.avg("TOTAL_DELAY").alias("AVG_TOTAL_DELAY"),
    F.max("ARR_DELAY").alias("MAX_ARR_DELAY"),
    F.count("*").alias("FLIGHT_COUNT")
).orderBy("FL_DATE")

In [0]:
df_final.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("FL_DATE") \
    .saveAsTable(f"{schema_name}.{table_name}")